In [95]:
#%run upload_docs.ipynb
#%run database.ipynb

In [96]:
from pypdf import PdfReader
import os

def read_pdf(file_path):
    reader = PdfReader(file_path)
    text = ""
    for page in reader.pages:
        text += page.extract_text()
    return text

def load_pdfs(folder_path="data/documents"):
    """
    Reads all PDFs from data/documents folder and returns a list of texts.
    """
    docs = []
    for file in os.listdir(folder_path):
        if file.endswith(".pdf"):
            path = os.path.join(folder_path, file)
            docs.append(read_pdf(path))
    return docs

In [97]:
# embeddings.py
from sentence_transformers import SentenceTransformer

# Initialize model inside notebook
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

def create_embedding(text):
    return embedding_model.encode(text)

In [ ]:
# database.py
import psycopg2
import numpy as np

# Connect to your PostgreSQL database
conn = psycopg2.connect(
    host="localhost",
    database="rag_db",
    user="postgres",
    password="###############",  # replace with your actual password
    port="####" # replace with actual port number
)
cursor = conn.cursor()

def store_document(content, embedding):
    embedding_list = embedding.tolist()  # convert numpy array to list
    cursor.execute(
        "INSERT INTO documents (content, embedding) VALUES (%s, %s)",
        (content, embedding_list)
    )
    conn.commit()

def get_all_documents():
    cursor.execute("SELECT id, content, embedding FROM documents")
    rows = cursor.fetchall()
    return [(r[0], r[1], np.array(r[2], dtype='float32')) for r in rows]

In [99]:
#import nbimporter
#from upload_docs import load_pdfs
#from embeddings import create_embedding
#from database import store_document

# Load PDFs from data/documents
docs = load_pdfs()

for doc in docs:
    emb = create_embedding(doc)
    store_document(doc, emb)

In [ ]:
# Cell 4 — RAG retrieval with Gemini LLM
import numpy as np
#from database import get_all_documents
import google.generativeai as genai

# Configure Gemini LLM
genai.configure(api_key="#########################################") # replace with actual API key
llm_model = genai.GenerativeModel("gemini-2.5-flash")

# Load all documents from DB
documents = get_all_documents()

def retrieve(query, top_k=3):
    query_emb = create_embedding(query).astype('float32')
    def cosine_sim(a, b):
        return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))
    
    sims = [(doc[1], cosine_sim(query_emb, doc[2])) for doc in documents]
    sims.sort(key=lambda x: x[1], reverse=True)
    return [s[0] for s in sims[:top_k]]

def ask_llm(contexts, question):
    context_text = "\n".join(contexts)
    prompt = f"""
    Answer the question using the context below:
    {context_text}

    Question: {question}
    """
    response = llm_model.generate_content(prompt)
    return response.text

In [103]:
# Cell 5 — Test query
question = "What is Machine Learning?"
top_docs = retrieve(question)
answer = ask_llm(top_docs, question)
print("Answer:", answer)

Answer: Machine Learning is a key area of AI that involves algorithms allowing computers to learn from data.


In [102]:
# for m in genai.list_models():
#     if "generateContent" in m.supported_generation_methods:
#         print(m.name)